# CARBAMYLATION ANALYSIS

**Authors**: George Weston, Matteo Degiacomi, Hao Man

This Jupyter notebook features a pipeline aimed at facilitating the identification of lysines in proteins that may be carbamylated. The pipeline is divided in the following steps.

* 1. [Data Gathering](#gather)
    - 1.1. [Mine UNIPROT database](#uniprot)
    - 1.2. [Mine PDB and AlphaFold databases](#pdb)
* 2. [Feature Extraction](#measure)
* 3. [Data Analysis](#analysis)
    - 3.1. [Loading and Cleaning the Data](#analysis_load)
    - 3.2. [Data Aggregation and Plotting](#analysis_aggregate)
    - 3.3. [GO Terms Extraction](#analysis_go)
* 4. [Interactive Data Visualization](#plot)

Further explanations about each of these steps are provided in the individual sections below.

Let's get started by executing the cell below (select it and press the <kbd>&#x21E7; Shift</kbd> + <kbd>&#x23ce; Enter</kbd>keys) to import all the Python packages required.

In [ ]:
import os

#data handling
import numpy as np
import pandas as pd
pd.set_option("display.max_rows", None, "display.max_columns", None)

#plotting
import plotly.express as px
import plotly.graph_objects as go
import nglview as nv
from ipywidgets import HBox, VBox
from ipywidgets import widgets

#carbamylation analysis procedures
from uniprot import Uniprot
from protein import PDB
from measure import Measure
from analysis import Analysis
from viewer import Viewer

Before getting started, we should also define an output directory. This will be the location for all output files created throught this code. 

If it is not defined, or none is passed, the default name will be "result".

In [ ]:
outdir = "Demo"

## 1. Data Gathering  <a class="anchor" id="gather"></a>

### 1.1. Mine UNIPROT database <a class="anchor" id="uniprot"></a>

We will start by compiling a database of protein data of interest by mining the [UNIPROT database](https://www.uniprot.org/). This can be done by either providing an input text file, or by directly providing UNIPROT codes. Methods to automatically fetch UNIPROT codes for a whole organism are available.

The main commands enabling information gathering to compile the database are `Uniprot.from_csv_file` and `Uniprot.get_protein_data`. These can be launched multiple times, to compile a database saved in a pandas DataFrame (`Uniprot.df`).

Generally, you will only use one of the following methods at once, however multiple can be used and the results are appended onto each other.

#### 1.1.1. Selection using csv files 

We will start by loading a list of UNIPROT codes contained in a short text file. Let's have a look at the file itself before loading it into our pipeline.

In [ ]:
df = pd.read_csv(f'input{os.sep}cyano_reviewed_first10.csv')
df['Uniprot_Entry'] #show data





As you can see, the file simply contains a list of UNIPROT codes. Let's look for the data associated with them by mining UNIPROT!

In [ ]:
UP = Uniprot()
UP.from_csv_file(f'input{os.sep}cyano_reviewed_first10.csv')
#UP.from_csv_file(f'input{os.sep}old{os.sep}input_codes.csv')
UP.df #show data






#### 1.1.2. Selection via manual insertion 

Besides loading UNIPROT codes from a text file, we can also append hand picked proteins in the database, by providing their UNIPROT code. Every time the command `Uniprot.get_protein_data`, data is appended to the DataFrame of previously gathered proteins.

In [ ]:
UP = Uniprot()
UP.get_protein_data('Q03513')
UP.df #show data

#### 1.1.3. Whole organism selection

If you are interested in fetching all the UNIPROT codes associated with a specific organism, the methods `Uniprot.get_organism_proteins` is available. It takes as parameter a proteome ID, and returns a list of UNIPROT codes. The list can be then iterated on, to load information on each identified protein. Optionally, the parameter `reviewed_only=True` can be used to ensure only codes of reviewed proteins are retrieved. Before calling this method though, it may be a good idea to determine how many proteins codes will be returned, usin the method `Uniprot.get_organism_proteins`.

*What follows is an example. In the rest of this tutorial, we will work on the data previously fetched*.

We will look for all the proteins in the rabbit's proteome, code: [UP000001811](https://www.uniprot.org/proteomes/UP000001811).  Let's start by determining how many proteins are associated with this organism, and how many of these are reviewed.

In [ ]:
UP_organism = Uniprot()
count_all_prot = UP_organism.count_organism_proteins('UP000001811')
count_reviewed_prot = UP_organism.count_organism_proteins('UP000001811', reviewed_only=True)
print(f'total: {count_all_prot}, reviewed: {count_reviewed_prot}')

Unsurprisingly, only a fraction of all proteins are reviewed. Let's only gather these ones. The `Uniprot.get_organism_proteins` method will return a list of relevant UNIPROT codes, and print progress text while it runs (i.e., "number of fetched UNIPROT codes / total number of codes to fetch").

In [ ]:
data = UP_organism.get_organism_proteins('UP000001811', reviewed_only=True)

If you want to see how infromation is retrieved from a list of UNIPROT codes, uncomment the lines below.
If not, there is no need to run this cell.

<div class="alert alert-warning">
<b>Warning:</b> For a whole organism, incuding unreviewed proteins, this can take a long time!
</div>

In [ ]:
#for cnt, d in enumerate(data):
#    print(f'{cnt}/{len(data)}: {d}')
#    UP_organism.get_protein_data(d)    

### 1.2. Mine PDB and AlphaFold databases  <a class="anchor" id="pdb"></a>

After gathering a list of Uniprot codes (contained in the Uniprot DataFrame - `Uniprot.df`), conformations for all proteins described need to be downloaded and curated.
Files will be downloaded from the [Protein Data Bank (PDB)](https://www.rcsb.org/) and the [AlphaFold database](https://alphafold.ebi.ac.uk/).
Two optional parameters can be passed to the PDB constructor, `outdir` (output directory, "result" by default) and the largest acceptable gap in sequence (`gap`, 10 by default).
* `outdir` has been set in a cell higher up in the tutorial
* `gap` is used when conformations are incomplete to give a maximum distance that Modeller will patch conformations with

Proteins having a consecutive number of missing aminoacids greater than this cutoff value will be disregarded.

For each entry in the input DataFrame:
* raw files from the PDB and Alphafold databases are downloaded in a subfolder within the output directory (defined by `PDB.raw_dir`, "conformations" by default)
* each raw file is cleaned: heteroatoms, hydrogens, and non-protein non-standard aminoacids are amended. Ions are preserved, as they may have important effect on the later feature extraction phase
* different conformations (usually from NMR ensembles) are saved in different files
* if multiple aminoacid rotamers exist, multiple files are produced (one per rotamer)
* all produced files are patched, i.e. missing parts in the sequence are added using Modeller
* all files correctly patched are saved in a different subfolder in the output directory (defined by `PDB.curated_dir`, "curated" by default).

This process will generate a new DataFrame, `PDB.df`, containing information on all the structures that have been correctly curated.

<div class="alert alert-warning">
<b>Warning:</b> depending on how many proteins have to be downloaded and curated, this operation may take a long time. AlphaFold and PDB files not needing curation are processed in few seconds. Large multimers requiring curation can take up to 1 min. Multiple conformations in a single protein structure are individually curated.
</div>

In [ ]:
pdb = PDB(outdir=outdir, gap=10)
pdb.gather_proteins(UP.df)

Show the DataFrame reporting on structures successfully downloaded and curated, and save it into a CSV file. 

A filename can be given via the `outname` option (eg `pdb.save_state(outname = proteins.csv)`). If no filename is given, file `proteins.csv` will be saved in the output directory.

In [ ]:
pdb.save_state()
pdb.df #show data

If you have already gathered protein data and wish to load the CSV file with the protein information to continue, this can be done with the method `PDB.load_state(filename)`.

In [ ]:
pdb = PDB(outdir=outdir, gap=10)
pdb.load_state(f'{outdir}{os.sep}proteins.csv')
pdb.df #show data

## 2. Feature Extraction  <a class="anchor" id="measure"></a>

After structures have been curated through the previous steps within this workbook, features can be calculated for the lysines within the proteins.
The `Measure` class is used for this, when using this a log file will be created within `outdir`. The filename is defined by the parameter `log_path`, and is "measure_log.txt" by default.
The following features are currently available for calculation:

<style>
.table_component {
    overflow: auto;
    width: 100%;
}

.table_component table {
    border: 1px solid #dededf;
    height: 100%;
    width: 100%;
    table-layout: auto;
    border-collapse: collapse;
    border-spacing: 1px;
    text-align: left;
}

.table_component caption {
    caption-side: top;
    text-align: left;
}

.table_component th {
    border: 1px solid #dededf;
    background-color: #eceff1;
    color: #000000;
    padding: 5px;
}

.table_component td {
    border: 1px solid #dededf;
    background-color: #ffffff;
    color: #000000;
    padding: 5px;
}
</style>
<div class="table_component" role="region" tabindex="0">
<table>
    <caption><br></caption>
    <thead>
        <tr>
            <th>Feature Name<br></th>
            <th>Code Name<br></th>
            <th>Details</th>
        </tr>
    </thead>
    <tbody>
        <tr>
            <td>pKa (PROPKA3)<br></td>
            <td>propka</td>
            <td>Calculates the pKa of the lysine site using the PROPKA3 software.</td>
        </tr>
        <tr>
            <td>pKa (pkaANI)<br></td>
            <td>pkaANI</td>
            <td>Calculates the pKa of the lysine site using the pkaANI software.</td>
        </tr>
        <tr>
            <td>Depth</td>
            <td>depth</td>
            <td>Calculates the minimum depth of the NZ atom of the lysine from the surface of the protein using the depth method from biobox.</td>
        </tr>
        <tr>
            <td>Solvent Accessible Surface Area (SASA)<br></td>
            <td>sasa</td>
            <td>Calculates the sovlvent accessible surface area (SASA) of the NZ atom of the lysine within the protein using the sasa method from biobox.</td>
        </tr>
        <tr>
            <td>Atomic Environment Vectors (AEVs)<br></td>
            <td>aev</td>
            <td>Uses the ANI-2x software to produce atomic environment vectors (AEVs) to represent both local environment of the NZ atom within the lysines. This encodes the structure and the neighbouring atoms into a 1008 value vector.</td>
        </tr>
        <tr>
            <td>Dynamically Accessible Surface (DAS)<br></td>
            <td>das</td>
            <td>Calculates the dynamically accessible surface (DAS) area of the NZ atom of the lysine within the protein using the __get_half_spheres() method from the Xlink class in biobox.</td>
        </tr>
    </tbody>
</table>
</div>

To define which features to calculate, use the features option, eg `features=['propka', 'pkaANI', 'sasa', 'depth', 'aev', 'das']`. By default, all of the features are calculated. 

The final results are saved into a CSV file. The name can be defined using the `outname` option, if no filename is given, `measures.csv` is used. This is saved into a folder defined by `outdir` ("result" by default). 

<div class="alert alert-warning">
<b>Warning:</b>  measurement calculations typically take anywhere between 2 seconds and 10 minutes per curated protein structure. The time depends on the structure size and the number of lysines within the protein.
</div>

In [ ]:
M = Measure(pdb.df, outdir=outdir, activate_log=True, log_path="measure_log.txt")
M.measure_dataframe()
M.save_state()

<div class="alert alert-info">
<b>Note</b>: the <code>Measure</code> object features methods for calculations and data recovery:
    
* <code>M.restart_measure()</code> restarts the operations of <code>M.measure_dataframe()</code>, if these were interrupted by any reason.
* <code>M.recover_from_log(filename)</code> parses a logfile and returns a DataFrame.</div>


The data acquired during calculations is located in the `M.df` DataFrame. We can sort the data according to any key of interest, for instance UNIPROT entry followed by resid.

In [ ]:
M.df.sort_values(['Uniprot_Entry', 'Resid'])

## 3. Data Analysis  <a class="anchor" id="analysis"></a>

### 3.1. Loading and cleaning the data   <a class="anchor" id="analysis_load"></a>

The Analysis class is responsible for interpreting the lysine features we extracted. Creating `Analysis` requires a DataFrame containing data gathered by a `Measure` instance. This can come either from a previously saved logfile (as we shall do in the next box), or from a DataFrame in a `Measure` instance (`Measure.df`). Below, we will load data from the CSV file file we previously saved.

In [ ]:
df = pd.read_csv(f'data{os.sep}measures_cut.csv')

#To see a more complete dataset, uncomment the following line to load a much larger pre-calculated dataset 
#df = pd.read_csv(f'all_cyano{os.sep}training_data_all.csv')

analysis = Analysis(df)

The loaded measurement data is now internally stored in the `Analysis.df` DataFrame.

Some lysine measurements may be suspiciously high (e.g., pKa>14) or low (e.g., pKa<1). This could be an indication that, in their specific circumstance, calculations were faulty. These can be identified as follows, where the output is a new DataFrame.

In [ ]:
df_suspicious = analysis.get_extreme_values(feature='propka', lower=1, upper=14)

Having identified suspicious measurements, we can delete them from our `Analysis.df` dataset as follows.

In [ ]:
analysis.remove_df(df_suspicious)

### 3.2. Data Aggregation and Slicing   <a class="anchor" id="analysis_aggregate"></a>

#### 3.2.1 Aggregation

The data we have now cleaned can be aggregated. Aggregation involves calculating min, max, mean and standard deviation for each feature (a single lysine may feature multiple measurements in the input DataFrame). This operation is carried out by calling `Analysis.df_aggregate()`, and the data is internally stored in the DataFrame `Analysis.df_aggregated`.

In [ ]:
analysis.aggregate()
analysis.df_aggregated #show data

An alternative method to aggregate data is, for each lysine, to only retain one representative measurement of pKa and SASA. The selection criterion can either be `average` or `south_east`. The `south_east` option is able to evaluate the trade off between metrics. Either enter 2 or 3 metrics to analyse the trade off between (lowest pKa, highest SASA and lowest depth) and insert these as options within metrics as a list eg. ['pKa', 'depth', 'sasa']. Weights can be given for each metric to see which to prioritise and are given as input in a list in the same way, the order of weights corresponds to the order of the metrics: eg. weight=[0.5, 0.8, 0.3]. The default weight is even for each. The result of this aggregation will be stored in the DataFrame `Analysis.df_sub`.

In [ ]:
analysis.subset(analysis.df, method = 'south_east', metrics=['propka', 'depth', 'sasa'], weight=[0.5, 0.8, 0.3])
analysis.df_sub #show data

#### 3.2.2 AlphaFold vs PDB data

An interesting question involves comparing AlphaFold data to PDB data. The following method enables selecting only the data from structures predicted by AlphaFold. Note that any DataFrame in `Analysis` can always be interrogated with pandas syntax, e.g., via the [query](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.query.html) method.

In [ ]:
df_af = analysis.get_data_alphafold()
df_af #show data

#### 3.2.3 Basic Plotting of Data

`Analysis` features a method to produce basic plots of the data (histogram and boxplot). Note that if no UNIPROT code and/or resid id is given, the plot will use the data of the whole DataFrame.

In [ ]:
analysis.plot_graph(plot_type = 'histogram', 
                    feature = 'propka', 
                    uniprot_entry = False, # can also specify a uniprot code and a resid ID
                    resid = False)

In [ ]:
analysis.df.plot.scatter("sasa", "propka")

### 3.3. GO Terms Extraction  <a class="anchor" id="analysis_go"></a>

Every protein is associated to a set of [Gene Ontology (GO) Terms](http://geneontology.org/). `Analysis` enables scraping these terms from the UNIPROT database. The data will be stored in a dictionary inside the `Analysis` class. *Technical note: this operation uses multi-threading for speed up.*

In [ ]:
analysis.GO_get_data()

Once the GO Terms of all the analysed proteins have been scraped, they can be interrogated with two different methods.

The first method searches for all the protein structures matching a specific GO code. Below, we will look for all lysines belonging to proteins associated with the GO term [0042803](https://www.ebi.ac.uk/QuickGO/term/GO:0042803) ("*dimerization activity*").

In [ ]:
analysis.GO_search_term(analysis.df, code ='0042803', name ='')

The second method extracts the GO Terms of a specific protein, defined by its UNIPROT code. Below, we retrieve all the GO termns of the protein P73832.

In [ ]:
GO_terms = analysis.GO_search_protein('P06733')
GO_terms #show data

We will now look for enrichment of GO terms within a specific pKa and SASA region. This enables us to determine whether specific GO terms are more prevalent in specific regions of the pKa/SASA space.

In [ ]:
pka_range = [1.0, 9.3]
sasa_range = [21.9, 100.0]
df = analysis.enrichment_analysis(pka_range, sasa_range)
df # show data

## 4. Interactive Data Visualization  <a class="anchor" id="plot"></a>

Once we have scraped all the GO Terms, we pass our `Analysis` object to a new class: `Viewer`. Let's start by initialising it. To do so, an additional parameter is required: `outdir`. This informs viewer about where the protein structures associated with the data in `Analysis` are located (having protein structures available is a requirement).

In [ ]:
cp = Viewer(analysis, outdir=outdir)

Once initialised, `Viewer` offers the `advanced_plot` method, usable in the following ways:

* without selecting any region, one can examine a GO Term shown in the dropdown lists and then click one of the red points (these are the average or the `south_east` measurement of the lysines that come from the proteins that have the GO Term). Finally, click a green point (an actual measurement of a lysine based on a particular PDB file) to see the 3D structure.
* one can also first select a region using the slides of pka and sasa. Note this operation will automatically initiate enrichment analysis. A bar chart will be produced based on the results of enrichment analysis. Finally, one can carry on the same process discussed above.
* if a region is selected, the export button can save the data (including GO Terms) within the region to a csv file and store it in the outdir defined by the user
* if a region is selected, the reset button can clear the region and the results of enrichment analysis
* if a 3D visualisation is displayed, the clear button can clear it.

The `cutoff` parameter is optional, and helps defining which GO terms entries should be ignored in bar charts (do not show any case appearing in the region of interest less or equal than this number of times, default value is 0).

In [ ]:
cp.advanced_plot(export_path='Regional_Data.csv', cutoff=0)

---